<a href="https://colab.research.google.com/github/nayakamhrudaya/GenAI/blob/main/CB1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# INSTALL
!pip install faiss-cpu pypdf openai numpy

import numpy as np
import faiss
from openai import OpenAI
from pypdf import PdfReader
from google.colab import files

# 🔑 API KEY (direct)
client = OpenAI(api_key="Generted_Api_key")

# =========================
#  USER UPLOAD FILE
# =========================
uploaded = files.upload()

filename = list(uploaded.keys())[0]
print("Uploaded file:", filename)

# =========================
# 📄 READ FILE (PDF + TXT + MD)
# =========================
def read_file(filename):
    text = ""

    if filename.endswith(".txt") or filename.endswith(".md"):
        with open(filename, "r", encoding="utf-8") as f:
            text = f.read()

    elif filename.endswith(".pdf"):
        pdf = PdfReader(filename)
        for page in pdf.pages:
            text += page.extract_text() or ""

    return text

text = read_file(filename)
print("File loaded successfully!")

# =========================
#  IMPROVED CHUNKING
# =========================
def chunk_text(text, chunk_size=300):
    words = text.split()
    return [" ".join(words[i:i+chunk_size]) for i in range(0, len(words), chunk_size)]

chunks = chunk_text(text)

# =========================
#  EMBEDDINGS
# =========================
def embed(text):
    res = client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )
    return np.array(res.data[0].embedding, dtype=np.float32)

# batch embeddings (faster + safer)
vectors = []
for c in chunks:
    vectors.append(embed(c))

vectors = np.array(vectors)

# =========================
#  FAISS INDEX
# =========================
index = faiss.IndexFlatL2(vectors.shape[1])
index.add(vectors)

print("Vector DB created!")

# =========================
#  SEARCH FUNCTION
# =========================
def search(query, k=3):
    q_vec = embed(query).reshape(1, -1)
    _, idx = index.search(q_vec, k)
    return [chunks[i] for i in idx[0]]

# =========================
#  GPT ANSWER FUNCTION
# =========================
def ask(question):
    context = "\n\n".join(search(question))

    prompt = f"""
You are a helpful assistant.
Answer ONLY using the context below.

Context:
{context}

Question:
{question}
"""

    res = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}]
    )

    return res.choices[0].message.content

# =========================
#  CHAT LOOP
# =========================
while True:
    q = input("Ask (type exit to stop): ")

    if q.lower() == "exit":
        break

    print("\nAnswer:", ask(q))
    print("-" * 50)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 14.7 MB/s eta 0:00:00
